Importing required libraries:

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import pickle
import numpy as np
import os

In [2]:
from google.colab import files
uploaded = files.upload()

Saving blue_castle.txt to blue_castle.txt


Load and Pre-Process the data

In [3]:
file = open("blue_castle.txt", "r", encoding = "utf8" )

#store file in list
lines = []
for i in file:
     lines.append(i)

#Convert list to string
data = ""
for i in lines:
    data = '  '. join(lines)

#replace unnecessary stuff with space
data = data.replace('\n', '').replace('\r', '').replace('\ufeff', '').replace('“','').replace('”','')  #new line, carriage return, unicode character --> replace by space

#remove unnecessary spaces
data = data.split()
data = ' '.join(data)
data[:500]


'The Project Gutenberg eBook of The Blue Castle, by Lucy Maud Montgomery This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.org. If you are not located in the United States, you will have to check the laws of the country where you are located before usi'

In [4]:
len(data)

402906

Apply tokenization and some other changes

In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

#saving the tokenizer for predict function
pickle.dump(tokenizer, open('token.pkl','wb'))

sequence_data = tokenizer.texts_to_sequences([data])[0]
sequence_data[:15]

[1, 112, 97, 587, 4, 1, 94, 147, 58, 2383, 2384, 1818, 51, 587, 42]

In [7]:
len(sequence_data)

72052

In [8]:
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

8413


In [9]:
sequences = []

for i in range (3, len(sequence_data)):
  words = sequence_data[i-3:i+1]
  sequences.append(words)

print("The Length if sequences are:" , len(sequences))
sequences = np.array(sequences)
sequences[:10]

The Length if sequences are: 72049


array([[   1,  112,   97,  587],
       [ 112,   97,  587,    4],
       [  97,  587,    4,    1],
       [ 587,    4,    1,   94],
       [   4,    1,   94,  147],
       [   1,   94,  147,   58],
       [  94,  147,   58, 2383],
       [ 147,   58, 2383, 2384],
       [  58, 2383, 2384, 1818],
       [2383, 2384, 1818,   51]])

In [10]:
X = []
y = []

for i in sequences:
  X.append(i[0:3])
  y.append(i[3])

X = np.array(X)
y = np.array(y)

In [11]:
print("Data:", X[:10])
print("Response:", y[:10])

Data: [[   1  112   97]
 [ 112   97  587]
 [  97  587    4]
 [ 587    4    1]
 [   4    1   94]
 [   1   94  147]
 [  94  147   58]
 [ 147   58 2383]
 [  58 2383 2384]
 [2383 2384 1818]]
Response: [ 587    4    1   94  147   58 2383 2384 1818   51]


In [12]:
y = to_categorical(y, num_classes=vocab_size)
y[:5]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

Creating the model

In [20]:
model = Sequential()
model.add(Embedding(vocab_size, 10, input_length=3))
model.add(LSTM(1000, return_sequences=True))
model.add(LSTM(1000))
model.add(Dense(1000, activation= "relu"))
model.add(Dense(vocab_size, activation="softmax"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Train the model

In [23]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint("next_word.h5", monitor='loss', verbose=1, save_best_only=True)
model.compile(loss="categorical_crossentropy", optimizer=Adam(learning_rate=0.001))
model.fit(X, y, epochs=20, batch_size=64, callbacks=[checkpoint])

Epoch 1/20
1124/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 7.0318
Epoch 1: loss improved from inf to 6.77553, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 35s 24ms/step - loss: 7.0311
Epoch 2/20
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 6.2948
Epoch 2: loss improved from 6.77553 to 6.25814, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 39s 24ms/step - loss: 6.2948
Epoch 3/20
1125/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 5.8817
Epoch 3: loss improved from 6.25814 to 5.86224, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - loss: 5.8817
Epoch 4/20
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 5.5790
Epoch 4: loss improved from 5.86224 to 5.55682, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - loss: 5.5790
Epoch 5/20
1124/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 5.2846
Epoch 5: loss improved from 5.55682 to 5.29137, saving model to next_word.h5


...

1126/1126 ━━━━━━━━━━━━━━━━━━━━ 47s 29ms/step - loss: 2.4481
Epoch 17/20
1126/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 2.2257
Epoch 17: loss improved from 2.54060 to 2.32242, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 36s 24ms/step - loss: 2.2258
Epoch 18/20
1124/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 2.0009
Epoch 18: loss improved from 2.32242 to 2.10928, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - loss: 2.0012
Epoch 19/20
1124/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1.8291
Epoch 19: loss improved from 2.10928 to 1.91153, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 28s 25ms/step - loss: 1.8293
Epoch 20/20
1125/1126 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 1.6181
Epoch 20: loss improved from 1.91153 to 1.71468, saving model to next_word.h5


1126/1126 ━━━━━━━━━━━━━━━━━━━━ 28s 25ms/step - loss: 1.6183


Lets Predict

In [24]:
from tensorflow.keras.models import load_model
import numpy as np
import pickle

#Load the model and tokenizer
model = load_model('next_word.h5')
tokenizer = pickle.load(open('token.pkl', 'rb'))

def Predict_Next_Words(model, tokenizer, text):

  sequence = tokenizer.texts_to_sequences([text])
  sequence = np.array(sequence)
  preds = np.argmax(model.predict(sequence))
  predict_word = ""

  for key, value in tokenizer.word_index.items():
    if value == preds:
      predicted_word = key
      break

  print(predicted_word)
  return predicted_word

In [25]:
while(True):
  text = input("Enter your line:")

  if text == "0":
    print("Execution completed....")
    break

  else:
    try:
      text = text.split(" ")
      text = text[-3:]
      print(text)

      Predict_Next_Words(model, tokenizer, text)


    except Exception as e:
       print("Error occured: ",e)
       continue

Enter your line:My name
['My', 'name']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step
i
Enter your line:Hello 
['Hello', '']
Error occured:  Exception encountered when calling LSTM.call().

slice index 0 of dimension 1 out of bounds. for '{{node sequential_1_1/lstm_2_1/strided_slice}} = StridedSlice[Index=DT_INT32, T=DT_FLOAT, begin_mask=5, ellipsis_mask=0, end_mask=5, new_axis_mask=0, shrink_axis_mask=2](sequential_1_1/embedding_1_1/GatherV2, sequential_1_1/lstm_2_1/strided_slice/stack, sequential_1_1/lstm_2_1/strided_slice/stack_1, sequential_1_1/lstm_2_1/strided_slice/stack_2)' with input shapes: [1,0,10], [3], [3], [3] and with computed input tensors: input[1] = <0 0 0>, input[2] = <0 1 0>, input[3] = <1 1 1>.

Arguments received by LSTM.call():
  • sequences=tf.Tensor(shape=(1, 0, 10), dtype=float32)
  • initial_state=None
  • mask=None
  • training=False
Enter your line:the project is about 
['is', 'about', '']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
it


KeyboardInterrupt: Interrupted by user